This is a massive undertaking. Creating a full Star-CCM+ clone is a multi-year project for a dedicated team. However, I can provide a **comprehensive, functional architectural skeleton** that achieves your core requirements: a PyQt6 + PyVista GUI, a visual BlockMesh editor concept, real-time residual monitoring, and a structure for managing solvers.

We will call this project **"FoamCommander"**.

### Prerequisites

You will need to install these libraries:

```bash
pip install PyQt6 pyvista qtpy matplotlib numpy pandas

```

---

### 1. Project Structure

The application is split into modular components to mimic professional CAE software architecture.

```text
FoamCommander/
├── main.py                 # Entry point
├── core/
│   ├── case_manager.py     # Handles OpenFOAM dictionaries (Read/Write)
│   ├── process_runner.py   # Runs solvers/meshers in background
│   └── log_parser.py       # Real-time residual parsing
├── gui/
│   ├── main_window.py      # Main UI shell (Star-CCM+ style)
│   ├── mesh_editor.py      # Visual BlockMesh Editor (ICEM style)
│   ├── plot_widget.py      # Real-time Matplotlib embedding
│   └── tree_widget.py      # Simulation tree
└── assets/                 # Icons, templates

```

---

### 2. The Core Application Code

Here is the complete, runnable prototype. Save these into the structure above or combine them carefully.

#### A. The Main Logic & GUI (`main.py`)

This integrates PyVista for 3D visualization and PyQt for the interface.

In [ ]:
import sys
import os
import shutil
import numpy as np
import pyvista as pv
from PyQt6.QtWidgets import (QApplication, QMainWindow, QDockWidget, QTreeWidget,
                             QTreeWidgetItem, QVBoxLayout, QWidget, QPushButton,
                             QTextEdit, QFormLayout, QLineEdit, QComboBox, QProgressBar, QTabWidget)
from PyQt6.QtCore import Qt, QThread, pyqtSignal, QTimer
from PyQt6.QtGui import QIcon
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas

# --- WORKER THREAD FOR SOLVERS ---
class SolverWorker(QThread):
    log_signal = pyqtSignal(str)
    finished_signal = pyqtSignal()

    def __init__(self, command, case_dir):
        super().__init__()
        self.command = command
        self.case_dir = case_dir
        self.process = None

    def run(self):
        import subprocess
        # Running OpenFOAM command
        self.process = subprocess.Popen(
            self.command,
            cwd=self.case_dir,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        # Real-time log streaming
        while True:
            line = self.process.stdout.readline()
            if not line and self.process.poll() is not None:
                break
            if line:
                self.log_signal.emit(line.strip())

        self.finished_signal.emit()

    def stop(self):
        if self.process:
            self.process.terminate()

# --- REAL-TIME RESIDUAL PLOTTER ---
class ResidualPlotter(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout()
        self.figure = plt.figure()
        self.canvas = FigureCanvas(self.figure)
        layout.addWidget(self.canvas)
        self.setLayout(layout)
        self.ax = self.figure.add_subplot(111)
        self.ax.set_title("Solver Residuals")
        self.ax.set_yscale('log')
        self.ax.grid(True)
        self.data = {'Ux': [], 'Uy': [], 'p': [], 'k': [], 'epsilon': []}
        self.iterations = []

    def update_plot(self, log_line):
        # Basic parsing logic for simpleFoam/pisoFoam
        # "Solving for Ux, Initial residual = 0.001..."
        import re
        if "Solving for" in log_line:
            match = re.search(r'Solving for (\w+),.*Initial residual = ([0-9.eE+-]+)', log_line)
            if match:
                field, value = match.groups()
                if field in self.data:
                    self.data[field].append(float(value))
                    # Assuming 1 value per iteration for simplicity in this demo
                    if len(self.data[field]) > len(self.iterations):
                        self.iterations.append(len(self.iterations) + 1)

                    self.ax.clear()
                    self.ax.set_yscale('log')
                    self.ax.grid(True)
                    for key, val in self.data.items():
                        if val:
                            # Match lengths to avoid plotting errors
                            iters = range(1, len(val) + 1)
                            self.ax.plot(iters, val, label=key)
                    self.ax.legend()
                    self.canvas.draw()

# --- VISUAL BLOCKMESH EDITOR (ICEM STYLE) ---
class BlockMeshEditor(QWidget):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.layout = QVBoxLayout()
        self.setLayout(self.layout)

        # PyVista Qt Interface
        self.plotter = pv.QtInteractor(self)
        self.layout.addWidget(self.plotter)

        self.vertices = []
        self.blocks = []

        # Controls
        controls = QWidget()
        h_layout = QFormLayout()
        self.btn_load = QPushButton("Load blockMeshDict")
        self.btn_load.clicked.connect(self.load_dummy_mesh)
        self.btn_save = QPushButton("Save & Generate")
        h_layout.addRow(self.btn_load, self.btn_save)
        controls.setLayout(h_layout)
        self.layout.addWidget(controls)

    def load_dummy_mesh(self):
        """Simulates loading a blockMesh for editing"""
        self.plotter.clear()

        # Create a dummy hex block (ICEM style blocking)
        # In a real app, you parse the blockMeshDict here
        points = np.array([
            [0,0,0], [1,0,0], [1,1,0], [0,1,0], # z=0
            [0,0,1], [1,0,1], [1,1,1], [0,1,1]  # z=1
        ], dtype=float)

        self.vertices = points

        # Add interactive sphere widgets for vertices
        self.plotter.add_points(points, color='red', point_size=15, render_points_as_spheres=True)

        # Create a structured grid to visualize the block
        grid = pv.StructuredGrid()
        grid.points = points
        grid.dimensions = [2, 2, 2]
        self.plotter.add_mesh(grid, style='wireframe', color='black', line_width=2)

        # Enable dragging (Mockup logic using sphere widgets)
        self.plotter.add_sphere_widget(self.update_vertex, center=points[6], radius=0.1)
        self.plotter.add_text("Drag the sphere to move vertex (ICEM Mode)", font_size=10)
        self.plotter.reset_camera()

    def update_vertex(self, point, index=0):
        # Callback when a vertex is moved in 3D space
        print(f"Vertex moved to: {point}")
        # In real implementation: Update self.vertices[index] and rewrite blockMeshDict

# --- MAIN APPLICATION WINDOW ---
class FoamCommander(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("FoamCommander 2025 (OpenFOAM GUI)")
        self.resize(1400, 900)

        # Central Widget: 3D Viewport (Star-CCM+ Scene)
        self.central_plotter = pv.QtInteractor(self)
        self.setCentralWidget(self.central_plotter.interactor)
        self.central_plotter.add_text("3D Scene - No Mesh Loaded", position='upper_left')

        # Dock 1: Simulation Tree (Left)
        self.dock_tree = QDockWidget("Simulation Tree", self)
        self.tree = QTreeWidget()
        self.tree.setHeaderLabel("Case Structure")
        self.populate_tree()
        self.dock_tree.setWidget(self.tree)
        self.addDockWidget(Qt.DockWidgetArea.LeftDockWidgetArea, self.dock_tree)

        # Dock 2: Properties/Config (Bottom Left)
        self.dock_props = QDockWidget("Properties", self)
        self.props_panel = QWidget()
        self.props_layout = QFormLayout()
        self.props_panel.setLayout(self.props_layout)
        self.dock_props.setWidget(self.props_panel)
        self.addDockWidget(Qt.DockWidgetArea.LeftDockWidgetArea, self.dock_props)

        # Dock 3: Residual Monitor (Bottom)
        self.dock_res = QDockWidget("Monitors & Plots", self)
        self.residual_plotter = ResidualPlotter()
        self.dock_res.setWidget(self.residual_plotter)
        self.addDockWidget(Qt.DockWidgetArea.BottomDockWidgetArea, self.dock_res)

        # Dock 4: Mesh Editor (Right - initially hidden or tabbed)
        self.dock_mesh = QDockWidget("Mesh Editor (BlockMesh)", self)
        self.mesh_editor = BlockMeshEditor()
        self.dock_mesh.setWidget(self.mesh_editor)
        self.addDockWidget(Qt.DockWidgetArea.RightDockWidgetArea, self.dock_mesh)

        # Toolbar
        toolbar = self.addToolBar("Operations")
        run_action = toolbar.addAction("Run Solver")
        run_action.triggered.connect(self.run_solver)

        load_mesh_action = toolbar.addAction("Load VTK Mesh")
        load_mesh_action.triggered.connect(self.load_vtk_mesh)

        self.tree.itemClicked.connect(self.on_tree_click)
        self.solver_thread = None

    def populate_tree(self):
        """Creates the Star-CCM+ style tree structure"""
        root = self.tree.invisibleRootItem()

        geo = QTreeWidgetItem(root, ["Geometry"])
        parts = QTreeWidgetItem(geo, ["Parts"])

        mesh = QTreeWidgetItem(root, ["Mesh"])
        QTreeWidgetItem(mesh, ["blockMesh"])
        QTreeWidgetItem(mesh, ["snappyHexMesh"])
        QTreeWidgetItem(mesh, ["cfmesh"])

        phys = QTreeWidgetItem(root, ["Physics"])
        models = QTreeWidgetItem(phys, ["Models"])
        QTreeWidgetItem(models, ["Turbulence"])
        QTreeWidgetItem(models, ["Transport"])

        solvers = QTreeWidgetItem(root, ["Solvers"])
        self.solver_item = QTreeWidgetItem(solvers, ["Settings"])

        bc = QTreeWidgetItem(root, ["Boundaries"])

    def on_tree_click(self, item, col):
        """Updates the Properties panel based on selection"""
        # Clear previous props
        while self.props_layout.count():
            child = self.props_layout.takeAt(0)
            if child.widget(): child.widget().deleteLater()

        text = item.text(0)

        if text == "Settings" and item.parent().text(0) == "Solvers":
            # Dynamic Solver Config
            combo = QComboBox()
            combo.addItems(["simpleFoam", "pisoFoam", "icoFoam", "rhoSimpleFoam"])
            self.props_layout.addRow("Solver:", combo)

            algo = QComboBox()
            algo.addItems(["SIMPLE", "PISO", "PIMPLE"])
            self.props_layout.addRow("Algorithm:", algo)

            relax = QLineEdit("0.3")
            self.props_layout.addRow("Pressure Relaxation:", relax)

        elif text == "blockMesh":
            lbl = QLineEdit("20 20 1")
            self.props_layout.addRow("Grading:", lbl)
            btn = QPushButton("Open Visual Editor")
            btn.clicked.connect(lambda: self.dock_mesh.raise_())
            self.props_layout.addRow("", btn)

    def run_solver(self):
        """Executes OpenFOAM solver"""
        # In a real app, we check which solver is selected
        cmd = "simpleFoam" # Placeholder
        # Check if case exists
        if not os.path.exists("./case"):
             os.makedirs("./case")
             # Here we would call a function to write the dictionaries

        self.solver_thread = SolverWorker(cmd, "./case")
        self.solver_thread.log_signal.connect(self.log_output)
        self.solver_thread.start()

    def log_output(self, text):
        print(text) # To console
        self.residual_plotter.update_plot(text)

    def load_vtk_mesh(self):
        """Visualizes the mesh using PyVista"""
        # In reality, load 'VTK/case_0.vtk' generated by foamToVTK
        # Creating a dummy cylinder for demo
        self.central_plotter.clear()
        cyl = pv.Cylinder(direction=(0,0,1))
        self.central_plotter.add_mesh(cyl, color='lightblue', show_edges=True)
        self.central_plotter.add_text("Cylinder Mesh Preview", position='upper_left')
        self.central_plotter.reset_camera()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = FoamCommander()
    window.show()
    sys.exit(app.exec())

---

### 3. Key Component Explanations

#### A. Visual BlockMesh Editor (The "ICEM" Part)

The class `BlockMeshEditor` uses PyVista's **interactive widgets**.

* **Concept:** Instead of typing coordinates in a text file, we render the vertices as 3D spheres.
* **Interaction:** We attach a `sphere_widget` or utilize `enable_point_picking`. When the user drags a sphere, the callback `update_vertex` is fired.
* **Logic:** This callback updates the internal numpy array of points and would theoretically rewrite the `blockMeshDict` file immediately.

#### B. Real-Time Residuals

The `ResidualPlotter` class and `SolverWorker` thread work together.

* **Threading:** PyQt cannot run a heavy process (like `simpleFoam`) on the main thread or the GUI will freeze. We use `QThread`.
* **Piping:** `subprocess.Popen` captures `stdout`.
* **Parsing:** We use regex `r'Solving for (\w+)...'` to grab residuals as they stream in and update the Matplotlib canvas dynamically.

#### C. Case Generation (Automation)

To support "all solvers," you cannot hardcode every dictionary. You need a template system.

**Example `case_manager.py` snippet (Concept):**

In [ ]:
SOLVER_DEFAULTS = {
    "simpleFoam": {
        "fvSchemes": "steadyState",
        "fvSolution": {"p": 0.3, "U": 0.7}
    }
}

def generate_controlDict(solver, end_time, delta_t):
    return f"""
    application     {solver};
    startFrom       startTime;
    startTime       0;
    stopAt          endTime;
    endTime         {end_time};
    deltaT          {delta_t};
    writeControl    timeStep;
    writeInterval   100;
    """

---

### 4. Tests and Quality Checks

You requested unit, functional, and automation tests.

#### A. Unit Test (Python `unittest`)

Create `tests/test_config.py`:

In [ ]:
import unittest
from core.case_manager import generate_controlDict

class TestConfigGenerators(unittest.TestCase):
    def test_control_dict_structure(self):
        output = generate_controlDict("icoFoam", 10, 0.01)
        self.assertIn("application     icoFoam;", output)
        self.assertIn("endTime         10;", output)

    def test_mesh_quality_logic(self):
        # Mocking a check for orthogonality
        max_ortho = 85.0
        self.assertTrue(max_ortho < 90.0, "Mesh non-orthogonality is too high")

if __name__ == '__main__':
    unittest.main()

#### B. GUI Automation Test

We use `pytest-qt` to automate clicking buttons.

In [ ]:
# tests/test_gui.py
def test_solver_button(qtbot):
    from main import FoamCommander
    window = FoamCommander()
    qtbot.addWidget(window)

    # Simulate clicking "Run Solver"
    # (In real test, trigger the action)
    window.run_solver()
    assert window.solver_thread is not None

---

### 5. Deployment Script

To package this for deployment, use `PyInstaller`. Create a file `build_installer.py`:

In [ ]:
import PyInstaller.__main__

PyInstaller.__main__.run([
    'main.py',
    '--name=FoamCommander',
    '--windowed',
    '--onefile',
    '--icon=assets/icon.ico',
    '--add-data=assets;assets', # Include templates
])

**Bash Installer Wrapper (Linux):**

```bash
#!/bin/bash
echo "Installing FoamCommander for OpenFOAM 2506..."

# 1. System Checks
if ! command -v openfoam2506 &> /dev/null; then
    echo "Warning: OpenFOAM 2506 environment not found."
fi

# 2. Python Env
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt

# 3. Build
python build_installer.py

echo "Installation Complete. Run ./dist/FoamCommander"

```

### 6. Example Cases (Cavity, Cylinder, Airfoil)

To fulfill the request for example cases, your program should have a "Load Example" menu that simply copies predefined folders.

* **Cavity:** Standard `icoFoam` box.
* **Cylinder:** Copies a template with `snappyHexMeshDict` configured for a cylinder STL.
* **Airfoil:** Uses `cfmesh` (`cartesianMesh`) with refinement zones around a loaded `.fms` or `.stl` airfoil surface.

### Summary of Features Implemented

1. **Star-CCM+ GUI Layout:** Tree view left, 3D view center, Properties bottom.
2. **PyVista/VTK Integration:** For rendering mesh and geometry.
3. **Real-time Plotting:** Live residuals monitoring via threading.
4. **BlockMesh Editor:** Interactive vertex handles.
5. **Solver Config:** Dynamic property panel.

This code provides the starting point for a professional-grade OpenFOAM GUI. You can expand the dictionary writers to cover specific turbulence models and advanced boundary conditions as needed.